# Експеримент B: чи може CTI дати більше, ніж порт? (UNSW-NB15 + CIC-IDS2017)

**Запуск:** Run All. Під час роботи оновлюється один рядок прогресу, а наприкінці друкується **один підсумковий звіт** — його й треба скопіювати. Тривалість ≈1.5–2.5 год. Прогнози моделей кешуються, тож після збою повторний запуск не перенавчає готові моделі.

## Теоретична основа (записано ДО запуску)
Поточні CTI-ознаки сталі всередині сервісу, тобто Z = g(port). Звідси I(Z; Y | port) = 0: самостійно вони не можуть дати нової інформації понад порт. Ми це вже підтвердили емпірично.

Тому перевіряємо дві речі.
1. **Відхилення від профілю сервісу (DEV).** Для кожного сервісу на train рахується «нормальний» профіль, тобто медіана і IQR ознак F0 лише на BENIGN-потоках train. Ознака показує, наскільки потік відхиляється від профілю свого сервісу (робастні z-оцінки, обрізані до ±50), плюс середнє і максимум |z|. Ці ознаки змінюються всередині сервісу.
2. **CTI-зважування відхилень.** Індекс ризику сервісу r(s) — середній перцентильний ранг (серед сервісів) усіх числових CTI-ознак, що не є константами в датасеті. Ознаки: r(s), dev_mean·r(s), dev_max·r(s).

Чесне очікування з теорії: r·dev — це функція (dev, сервіс), тож нової інформації понад M5 вона також не несе. Можливий лише ефект представлення (деревам із num_leaves=15 легше вивчити взаємодію). Експеримент це перевіряє.

## Моделі (кожна — 500 дерев, заморожені гіперпараметри, без early stopping)
| | Ознаки |
|---|---|
| M0 | F0 |
| M1 | F0 + порт (число) |
| M2 | F0 + порт (категорія: топ-200 портів train + «other») |
| M3 | F1 — поточний CTI |
| M4 | F0 + сервіс (cve_ctx_service, групування від CTI) |
| M5 | M4 + DEV |
| M6 | M5 + CTI-зважування — **запропонований варіант** |
| M7 | M5 + placebo: ті самі ознаки, але r(s) переставлено між сервісами |

## Спліти
- **UNSW-NB15:** хронологічний спліт статті (Stime, 0.70/0.85).
- **CIC-IDS2017:** виправлені таймстемпи (AM/PM), стратифікований часовий спліт (день × клас, 70/15/15).

## Гіпотези і критерії (95% блочний bootstrap, блоки = хвилини, 500 реплік)
- **Первинна P1:** M6 − M5 > 0, тобто CTI-зважування додає щось понад відхилення від профілю сервісу. Критерій: Holm-скоригований односторонній p < 0.05 по двох датасетах **і** нижня межа CI > 0.
- **Вторинні (звітуються всі, незалежно від результату):**
  - S1: M6 − M7 — справжній CTI-ризик проти placebo;
  - S2: M5 − M0 — чи допомагає саме відхилення від профілю сервісу (не CTI);
  - S3: M6 − M1 — запропонований варіант проти сирого порту;
  - S4: M4 − M2 — групування від CTI проти категоріального порту;
  - S5: M3 − M1 — повторне підтвердження: поточний CTI проти порту.
- **Метрики:** AP (основна), ROC-AUC, recall при FPR = 0.1%, частка прибраної залишкової помилки відносно M0.

Дизайн ознак зафіксовано заздалегідь і ні на чому не підбирався. Test-сети відкриваються один раз.

In [2]:
import os, gc, re, time, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
warnings.filterwarnings("ignore")

# ---------------- конфігурація ----------------
OUT = "/srv/ids_research/data/expB"
CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)
UNSW_PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
CIC_PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
SEED, ROUNDS, N_BOOT = 42, 500, 500
FPR_TARGET, TOP_PORTS, MIN_SVC_BENIGN, Z_CLIP, CHUNK = 0.001, 200, 50, 50.0, 400_000

F0_UNSW = ['state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','Sload','Dload','Spkts','Dpkts',
'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Sintpkt',
'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
'ct_dst_src_ltm','proto','service']
CTX_UNSW = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_epss_coverage',
            'cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CTX_CIC = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']

try:
    from IPython.display import display, Pretty
    _h = display(Pretty("старт…"), display_id=True)
    def progress(msg): _h.update(Pretty(f"[{time.strftime('%H:%M:%S')}] {msg}"))
except Exception:
    def progress(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

REPORT, ERRORS = [], []
def R(line=""): REPORT.append(line)
def safe(n): return re.sub(r"[^0-9A-Za-z_]", "_", n)

def parse_port(v):
    s = str(v).strip().lower()
    try: return float(int(s, 16)) if s.startswith("0x") else float(s)
    except ValueError: return np.nan

# ---------------- завантаження і спліти ----------------
def load_unsw():
    df = pd.read_parquet(UNSW_PATH); df.columns = [c.strip() for c in df.columns]
    df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
    y = pd.to_numeric(df["Label"], errors="coerce").astype(int).to_numpy()
    t = pd.to_numeric(df["Stime"])
    b1, b2 = t.quantile(0.70, interpolation="nearest"), t.quantile(0.85, interpolation="nearest")
    split = np.where(t < b1, 0, np.where(t < b2, 1, 2))
    block = (t.to_numpy() // 60).astype(np.int64)
    raw = df["dsport"].astype(str)
    port = raw.map({u: parse_port(u) for u in raw.unique()}).astype(float).to_numpy()
    return dict(name="UNSW-NB15", split_desc="хронологічний спліт статті (Stime 0.70/0.85)", df=df, y=y,
                split=split, block=block, port=port, F0=F0_UNSW, CTX=CTX_UNSW, CAT={"state", "proto", "service"})

def load_cic():
    df = pd.read_parquet(CIC_PATH); df.columns = [c.strip() for c in df.columns]
    y = (df["attack_label"].astype(str) != "BENIGN").astype(int).to_numpy()
    raw = df["Timestamp"]
    ts = pd.to_datetime(raw) if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
    ts = ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))
    grp = ts.dt.date.astype(str) + "|" + df["attack_label"].astype(str)
    tn = ts.astype("int64")
    b1 = tn.groupby(grp).transform(lambda s: s.quantile(0.70, interpolation="nearest"))
    b2 = tn.groupby(grp).transform(lambda s: s.quantile(0.85, interpolation="nearest"))
    split = np.where(tn < b1, 0, np.where(tn < b2, 1, 2))
    block = pd.factorize(ts.dt.floor("min"))[0].astype(np.int64)
    port = pd.to_numeric(df["destination_port"], errors="coerce").astype(float).to_numpy()
    return dict(name="CIC-IDS2017", split_desc="виправлені таймстемпи, стратифікований часовий спліт (день×клас 70/15/15)",
                df=df, y=y, split=split, block=block, port=port, F0=F0_CIC, CTX=CTX_CIC, CAT={"protocol"})

# ---------------- ознаки ----------------
def build_features(D):
    df, y, split, port = D["df"], D["y"], D["split"], D["port"]
    F0, CTX, CAT = D["F0"], D["CTX"], D["CAT"]
    tr = split == 0
    n = len(df)
    svc = df["cve_ctx_service"].astype(str)
    svc = svc.where(~svc.isin(["nan", "<NA>", "None"]), "none").to_numpy()
    codes, uni = pd.factorize(svc)
    num_cols = [c for c in F0 if c not in CAT]
    Xn = np.array(df[num_cols].to_numpy(dtype="float32", na_value=np.nan), dtype="float32", copy=True)
    Xn[~np.isfinite(Xn)] = np.nan

    # профіль сервісу: медіана і масштаб (IQR, інакше std, інакше 1) лише на BENIGN train
    ben = tr & (y == 0)
    def stats(rows):
        sub = Xn[rows]
        med = np.nanmedian(sub, axis=0)
        q75, q25 = np.nanpercentile(sub, [75, 25], axis=0)
        sc = q75 - q25
        sd = np.nanstd(sub, axis=0)
        sc = np.where(np.isfinite(sc) & (sc > 0), sc, np.where(np.isfinite(sd) & (sd > 0), sd, 1.0))
        return np.nan_to_num(med, nan=0.0), sc
    gmed, gsc = stats(ben)
    MED = np.tile(gmed, (len(uni), 1)).astype("float32"); SC = np.tile(gsc, (len(uni), 1)).astype("float32")
    own = 0
    for k in range(len(uni)):
        rows = ben & (codes == k)
        if rows.sum() >= MIN_SVC_BENIGN:
            MED[k], SC[k] = stats(rows); own += 1

    Z = np.empty_like(Xn)
    for s in range(0, n, CHUNK):
        e = min(n, s + CHUNK); c = codes[s:e]
        Z[s:e] = np.clip((Xn[s:e] - MED[c]) / SC[c], -Z_CLIP, Z_CLIP)
    absZ = np.abs(Z)
    dev_mean = np.nanmean(absZ, axis=1).astype("float32")
    dev_max = np.nanmax(np.where(np.isnan(absZ), -np.inf, absZ), axis=1)
    dev_max = np.where(np.isfinite(dev_max), dev_max, np.nan).astype("float32")
    del absZ, Xn

    # індекс CTI-ризику сервісу: середній перцентильний ранг неконстантних числових CTI-ознак
    num_ctx = [c for c in CTX if c != "cve_ctx_service"]
    tab = df[num_ctx].apply(pd.to_numeric, errors="coerce").groupby(svc).first()
    tab = tab.drop(index="none", errors="ignore")
    used = [c for c in num_ctx if tab[c].nunique(dropna=True) > 1]
    r_tab = tab[used].rank(pct=True).mean(axis=1) if used else pd.Series(np.nan, index=tab.index)
    rng = np.random.default_rng(SEED)
    valid = r_tab.dropna()
    r_perm = pd.Series(valid.to_numpy()[rng.permutation(len(valid))], index=valid.index)
    r = pd.Series(svc).map(r_tab).astype(float).to_numpy().astype("float32")
    rp = pd.Series(svc).map(r_perm).astype(float).to_numpy().astype("float32")

    # порт як категорія (топ-N портів train)
    top = pd.Series(port[tr]).dropna().value_counts().index[:TOP_PORTS].to_numpy()
    pc = np.full(n, "other", dtype=object)
    m = np.isin(port, top)
    pc[m] = port[m].astype(np.int64).astype(str)

    zcols = ["z_" + safe(c) for c in num_cols]
    X = pd.concat([
        df[F0 + CTX].reset_index(drop=True),
        pd.DataFrame(Z, columns=zcols),
        pd.DataFrame({"port_num": port, "port_cat": pc, "dev_mean_abs": dev_mean, "dev_max_abs": dev_max,
                      "r_cti": r, "devmean_x_r": dev_mean * r, "devmax_x_r": dev_max * r,
                      "r_plac": rp, "devmean_x_rp": dev_mean * rp, "devmax_x_rp": dev_max * rp}),
    ], axis=1)
    del Z
    cats = set(CAT) | {"cve_ctx_service", "port_cat"}
    for c in cats:
        X[c] = pd.Categorical(X[c], categories=pd.Categorical(X.loc[tr, c]).categories)

    DEV = zcols + ["dev_mean_abs", "dev_max_abs"]
    models = {
        "M0": ("F0", F0),
        "M1": ("F0+port(num)", F0 + ["port_num"]),
        "M2": ("F0+port(cat)", F0 + ["port_cat"]),
        "M3": ("F1 поточний CTI", F0 + CTX),
        "M4": ("F0+сервіс", F0 + ["cve_ctx_service"]),
        "M5": ("M4+DEV", F0 + ["cve_ctx_service"] + DEV),
        "M6": ("M5+CTI-ваги (запроп.)", F0 + ["cve_ctx_service"] + DEV + ["r_cti", "devmean_x_r", "devmax_x_r"]),
        "M7": ("M5+placebo-ваги", F0 + ["cve_ctx_service"] + DEV + ["r_plac", "devmean_x_rp", "devmax_x_rp"]),
    }
    info = dict(n_services=len(uni), own_profile=own, cti_used=used, n_dev=len(DEV))
    return X, cats, models, info

# ---------------- навчання, метрики, bootstrap ----------------
def recall_at_fpr(yt, p, fpr_t=FPR_TARGET):
    fpr, tpr, _ = roc_curve(yt, p)
    ok = fpr <= fpr_t
    return float(tpr[ok].max()) if ok.any() else 0.0

def run_dataset(D):
    name = D["name"]
    progress(f"{name}: побудова ознак")
    X, cats, models, info = build_features(D)
    y, split, block = D["y"], D["split"], D["block"]
    tr, te = split == 0, split == 2
    spw = (y[tr] == 0).sum() / y[tr].sum()
    params = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
                  min_data_in_leaf=50, verbosity=-1, scale_pos_weight=spw)
    y_te = y[te]
    preds = {}
    for i, (mid, (label, cols)) in enumerate(models.items(), 1):
        f = os.path.join(CACHE, f"{safe(name)}_{mid}_{int(te.sum())}_{len(cols)}.npy")
        if os.path.exists(f):
            preds[mid] = np.load(f); continue
        progress(f"{name}: навчання {mid} ({label}), {i}/{len(models)}, {len(cols)} ознак")
        cc = [c for c in cols if c in cats]
        mdl = lgb.train(params, lgb.Dataset(X.loc[tr, cols], label=y[tr], categorical_feature=cc), num_boost_round=ROUNDS)
        preds[mid] = mdl.predict(X.loc[te, cols]).astype("float64")
        np.save(f, preds[mid]); del mdl; gc.collect()

    met = {}
    for mid in models:
        p = preds[mid]
        met[mid] = dict(AP=average_precision_score(y_te, p), AUC=roc_auc_score(y_te, p), R=recall_at_fpr(y_te, p))
    base_err = 1 - met["M0"]["AP"]

    progress(f"{name}: блочний bootstrap")
    blk_te = block[te]
    _, inv = np.unique(blk_te, return_inverse=True)
    blocks = list(pd.Series(np.arange(len(inv))).groupby(inv).indices.values())
    rng = np.random.default_rng(SEED)
    comps = [("P1", "M6", "M5", "CTI-ваги понад DEV (первинна)"),
             ("S1", "M6", "M7", "справжній CTI-ризик vs placebo"),
             ("S2", "M5", "M0", "відхилення від профілю сервісу"),
             ("S3", "M6", "M1", "запропонований vs сирий порт"),
             ("S4", "M4", "M2", "CTI-групування vs порт(кат)"),
             ("S5", "M3", "M1", "поточний CTI vs сирий порт")]
    idx_samples = [np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))]) for _ in range(N_BOOT)]
    ci = {}
    for cid, a, b, desc in comps:
        pa, pb = preds[a], preds[b]
        pt = met[a]["AP"] - met[b]["AP"]
        vals = np.array([average_precision_score(y_te[j], pa[j]) - average_precision_score(y_te[j], pb[j]) for j in idx_samples])
        lo, hi = np.percentile(vals, [2.5, 97.5])
        ci[cid] = dict(a=a, b=b, desc=desc, d=pt, lo=lo, hi=hi, p=(1 + np.sum(vals <= 0)) / (1 + N_BOOT))

    # ---- звіт по датасету ----
    R("=" * 96); R(f"{name}: {D['split_desc']}")
    R(f"train/val/test = {int(tr.sum()):,} / {int((split == 1).sum()):,} / {int(te.sum()):,};  "
      f"частка атак у test = {y_te.mean():.4f} (= AP випадкового класифікатора);  scale_pos_weight = {spw:.4f}")
    R(f"сервісів: {info['n_services']}, з власним профілем BENIGN (≥{MIN_SVC_BENIGN} потоків train): {info['own_profile']};  "
      f"DEV-ознак: {info['n_dev']};  CTI-ознаки в індексі ризику r(s): {info['cti_used']}")
    R(""); R(f"{'модель':<4} {'опис':<24} {'AP':>9} {'ROC-AUC':>9} {'R@FPR0.1%':>10} {'ΔAP vs M0':>11} {'прибрано помилки':>17}")
    for mid, (label, _) in models.items():
        m = met[mid]; d = m["AP"] - met["M0"]["AP"]
        red = 100 * d / base_err if base_err > 0 else float("nan")
        R(f"{mid:<4} {label:<24} {m['AP']:>9.6f} {m['AUC']:>9.6f} {m['R']:>10.4f} {d:>+11.6f} {red:>16.1f}%")
    R(""); R(f"{'тест':<4} {'порівняння':<10} {'зміст':<34} {'ΔAP':>10} {'95% блочний CI':>26} {'p(1-бік)':>9}")
    for cid, c in ci.items():
        R(f"{cid:<4} {c['a'] + '−' + c['b']:<10} {c['desc']:<34} {c['d']:>+10.6f}   [{c['lo']:+.6f}, {c['hi']:+.6f}] {c['p']:>9.4f}")
    del X; gc.collect()
    return ci

# ---------------- запуск ----------------
t0 = time.time()
results = {}
for loader in (load_unsw, load_cic):
    try:
        progress(f"завантаження: {loader.__name__}")
        D = loader()
        results[D["name"]] = run_dataset(D)
        del D; gc.collect()
    except Exception as e:
        import traceback
        ERRORS.append(f"{loader.__name__}: {type(e).__name__}: {e}\n{traceback.format_exc()}")

# Holm для первинної гіпотези P1 по двох датасетах
R("=" * 96); R("ПЕРВИННА ГІПОТЕЗА P1 (M6 − M5 > 0), Holm по датасетах, α = 0.05")
p1 = sorted([(v["P1"]["p"], k, v["P1"]) for k, v in results.items()], key=lambda x: x[0])
mtot = len(p1); stop = False
for i, (p, k, c) in enumerate(p1):
    thr = 0.05 / (mtot - i)
    ok = (not stop) and p < thr and c["lo"] > 0
    if not (p < thr): stop = True
    R(f"  {k:<12} ΔAP={c['d']:+.6f}  CI [{c['lo']:+.6f}, {c['hi']:+.6f}]  p={p:.4f}  поріг Holm={thr:.4f}  → "
      f"{'ПІДТВЕРДЖЕНА' if ok else 'НЕ підтверджена'}")
R(""); R(f"Час виконання: {(time.time() - t0) / 60:.1f} хв;  ROUNDS={ROUNDS}, N_BOOT={N_BOOT}")
if ERRORS:
    R(""); R("ПОМИЛКИ:"); [R(e) for e in ERRORS]

report = "\n".join(REPORT)
with open(os.path.join(OUT, "expB_report.txt"), "w") as fh:
    fh.write(report)
progress("готово — звіт нижче (також збережено у expB_report.txt)")
print(report)

[14:54:30] готово — звіт нижче (також збережено у expB_report.txt)

UNSW-NB15: хронологічний спліт статті (Stime 0.70/0.85)
train/val/test = 1,441,568 / 308,915 / 308,935;  частка атак у test = 0.0821 (= AP випадкового класифікатора);  scale_pos_weight = 28.2135
сервісів: 28, з власним профілем BENIGN (≥50 потоків train): 14;  DEV-ознак: 40;  CTI-ознаки в індексі ризику r(s): ['cve_ctx_pool_size', 'cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches']

модель опис                            AP   ROC-AUC  R@FPR0.1%   ΔAP vs M0  прибрано помилки
M0   F0                        0.982439  0.998144     0.8028   +0.000000              0.0%
M1   F0+port(num)              0.985116  0.998132     0.8414   +0.002678             15.2%
M2   F0+port(cat)              0.985920  0.998390     0.8490   +0.003481             19.8%
M3   F1 поточний CTI           0.983854  0.998123     0.8218   +0.001415              8.1%
M4   F0+сервіс                 0.982976  0.998020     0.8206   +0.000538              3.1%
M5   M4+DEV                    0.983769  0.99817